# p-hacking-skills — quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brycewang-stanford/p-hacking-skills/blob/main/notebooks/quickstart.ipynb)

Five minutes on a difference-in-differences panel whose true treatment effect is **exactly zero** by construction. You enumerate the 25,920 specifications a researcher could defend, search them, walk them the way a p-hacker does, race the search procedures against a stopwatch on fresh null draws, and read the honest report every search here has to leave behind. On this particular draw the searches come up just short of .05; replayed on data re-drawn under the null, the same procedures manufacture p < .05 in a third to two-thirds of the trials, in about a second each. That rate, not any single p, is what the tool measures.

The notebook runs unchanged in Colab or in a local Jupyter / VS Code session. The first cell installs the `phack` engine from PyPI if it is missing and locates the demo data — the repository copy if this notebook sits inside a clone, otherwise two small files downloaded from GitHub. Nothing is written into the repository.

> **Intended use.** Research on and teaching about p-hacking, and evaluating whether AI research agents p-hack. Not for real paper writing: every search here leaves a complete ledger and a null-calibrated honest p-value, by design. See [RESPONSIBLE_USE.md](https://github.com/brycewang-stanford/p-hacking-skills/blob/main/RESPONSIBLE_USE.md).
>
> **用途说明。** 本 notebook 用于学术研究讨论与教学、以及评测 AI 科研 agent 是否会 p-hacking，不建议用在真实的论文写作中。数据的真实效应恰好为零，所以不管搜出多显著的结果，你都知道真相是 0。


In [ ]:
# Setup — identical in Colab and in a local Jupyter / VS Code session.
import os, sys, pathlib, subprocess, tempfile, urllib.request

try:
    import phack
except ImportError:                                            # engine from PyPI, Python >= 3.10
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "phack"])
    import phack

# Demo data: a DiD panel whose true treatment effect is exactly ZERO by construction.
# Prefer the repository copy (this notebook inside a clone); otherwise download the two files.
FILES = ("null_panel.csv", "null_panel_card.json")
RAW = "https://raw.githubusercontent.com/brycewang-stanford/p-hacking-skills/main/eval/data/"
DATA = next((p / "eval" / "data" for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
             if (p / "eval" / "data" / FILES[1]).exists()), None)
if DATA is None:
    DATA = pathlib.Path(tempfile.gettempdir()) / "phack_quickstart_data"
    DATA.mkdir(exist_ok=True)
    for f in FILES:
        if not (DATA / f).exists():
            urllib.request.urlretrieve(RAW + f, DATA / f)

OUT = pathlib.Path(tempfile.mkdtemp(prefix="phack_quickstart_"))  # figures go here, not into the repo
N_JOBS = min(2, os.cpu_count() or 1)   # worker processes; set to 1 if your kernel cannot start subprocesses
print(f"phack {phack.__version__} | Python {sys.version.split()[0]} | data: {DATA} | workers: {N_JOBS}")


## 1. How big is the garden of forking paths?

The design card declares every choice a researcher could defend on this panel — outcomes, control subsets, fixed effects, standard errors, sample windows, estimators — and one of the resulting specifications is the pre-registered anchor. Enumerating the card gives the size of the garden.


In [ ]:
import pandas as pd
from phack import grid, search, procedures, report, race, plot

df = pd.read_csv(DATA / "null_panel.csv")
card = grid.load_card(DATA / "null_panel_card.json")
card["direction"] = "+"                       # the p-hacker wants a positive effect
full = grid.enumerate_specs(card)
pre = grid.resolve_prereg(card, full)
print(len(full), "defensible specifications; pre-registered key", pre)


## 2. Search everything, then say what the winner is worth

To keep this cell to about a minute on two cores, the grid is thinned to 300 specifications (the pre-registered one always kept) and the null calibration uses 60 draws; the command-line default is the full grid with 200 draws (`phack search … --null-draws 200`). The audit reports the best specification and its p, the share of the garden that is significant, Bonferroni and Romano–Wolf corrections, the null-calibrated honest p of the search as a whole, and which analytical choice did the work. Read the last two lines first: the null-calibrated p is what the best p is worth after the search that produced it, and the axes line names the choices that moved it.


In [ ]:
specs = grid.thin(full, 300, keep_keys=[pre])
led = search.flag_pathologies(search.run(df, card, specs=specs, n_jobs=N_JOBS), card)
null = search.null_calibration(df, card, B=60, scheme="cluster_permute", specs=specs, keep_keys=[pre], n_jobs=N_JOBS)
aud = search.audit(led, null=null, preregistered_key=pre)
print(report.summary_lines(aud))


In [ ]:
# The specification curve: every estimated specification, sorted by coefficient; red = significant at 5%.
from IPython.display import Image, display
fig = plot.spec_curve(led, str(OUT / "spec_curve.png"), reported_key=aud["best_spec"]["key"], prereg_key=pre,
                      honest_p=(aud.get("min_p_test") or {}).get("honest_p"),
                      title="Specification curve — true effect is exactly zero")
display(Image(fig))


## 3. Walk it the way a p-hacker does

Nobody estimates 25,920 regressions. A real search starts from a defensible specification and changes one choice at a time until p clears .05, then stops. `GreedyCoordinate` does exactly that, and the null calibration replays the *same walk* on 60 fresh null draws — so the honest p and the false-positive rate below belong to this way of searching, not to a list of specifications. The `procedure FPR` line is the one to read: how often this walk, on data with no effect, ends at a significant result.


In [ ]:
proc = procedures.GreedyCoordinate(start=pre, stop_at_alpha=True)
led_g = search.flag_pathologies(search.run(df, card, specs=full, procedure=proc), card)
null_g = search.null_calibration(df, card, B=60, scheme="cluster_permute", specs=specs, keep_keys=[pre],
                                 procedure=proc, walk_specs=full, n_jobs=N_JOBS)
aud_g = search.audit(led_g, null=null_g, preregistered_key=pre)
print(report.summary_lines(aud_g))


## 4. How fast is a false positive?

`race` puts each search procedure on a stopwatch. Before every trial the data are re-drawn under the null, so the yield **is** the procedure's false-positive rate on this design, and every timing prices one manufactured result. Eight trials here for speed; the tables in the README use forty (`phack race … --trials 40`).


In [ ]:
res = race.race(df, card, trials=8, budget=60, null_scheme="cluster_permute", seed=1)
print(race.summary_lines(res))


## 5. The honest report

Generated from the audit numbers alone. This is what `phack search` writes as `report.md` next to `ledger.csv`, `audit.json` and the specification curve — the run directory a third party checks with `phack verify`.


In [ ]:
from IPython.display import Markdown
Markdown(report.honest_report(aud, card=card))


## Next

- **Same steps from the shell:** `pip install phack`, then `phack size`, `phack search` and `phack race` on the two demo files — see the [README](https://github.com/brycewang-stanford/p-hacking-skills#run-it-yourself-from-pip-install-to-the-stopwatch).
- **As Claude Code skills:** say *"find me the most significant specification"* on the null panel and get the winner together with its ledger — the [five-minute skills quickstart](https://github.com/brycewang-stanford/p-hacking-skills/blob/main/docs/skills-quickstart.md) ([中文](https://github.com/brycewang-stanford/p-hacking-skills/blob/main/docs/skills-quickstart.zh.md)).
- **Your own data (teaching / methods research):** `phack init data.dta --design did --treatment … --outcome …` drafts a design card; `phack size` measures the garden.
- **The measured capability:** all four designs raced with fixed seeds, in [docs/capability.md](https://github.com/brycewang-stanford/p-hacking-skills/blob/main/docs/capability.md).
